## Healthcare Institution Analysis in Almere

A notebook by Daniel Hoog

## 1. Problem Definition

**Mental health problems are common and increasing in the Netherlands.** According to the most recent national population study, NEMESIS-3 (2019–2022), 25.9% of people aged 18 to 75 met the criteria for at least one mental disorder in the past year [1]. This is not stable: the 12-month prevalence of all types of anxiety and mood disorders increased significantly between 2007–2009 and 2019–2022, with increases ranging from 56% to 125% [2]. Notably, the increase was stronger for students, younger adults (18–34 years) and city dwellers, and was not explained by the COVID-19 pandemic [3].

**Demand exceeds supply.** Access to care is a structural bottleneck. In 2025, people ran into a mental health waiting list more than 100,000 times, almost half of them waiting longer than the agreed maximum of 14 weeks, with an average wait of 24 weeks before treatment started [4].

**Almere scores worse than the national average on several mental health indicators.** The 2022 GGD Health Monitor for adults (18–64) shows the following for Almere compared to the Netherlands as a whole [5]:

| Indicator (adults 18–64, 2022) | Almere | Netherlands |
|---|---|---|
| Severely lonely | 21% | 15% |
| High risk of anxiety disorder or depression | 14% | 11% |
| Experienced (very) much stress in past 4 weeks | 32% | 25% |
| Moderate or severe psychological complaints | 13% | 9% |
| Serious suicidal thoughts in past 12 months (once to very often) | 15% | 12% |
| Difficulty making ends meet in past 12 months | 29% | 21% |

*Source: GGD Flevoland, Gezondheidsmonitor Volwassenen en Ouderen 2022 [5].*

> **Note:** these are municipal averages. Whether each difference is statistically significant is marked in the source table and should be checked there. Neighbourhood-level variation is not visible at this level, which is exactly the gap this analysis addresses.

**Almere is growing, and so will demand.** The city grew from nothing to about 230,000 inhabitants in roughly fifty years and is expected to continue growing [6]. Further expansion has been proposed: political parties have advocated building 25,000 homes in Almere Pampus to house 50,000 to 60,000 people [7]. Care facilities need to be planned with this growth in mind.

**Location matters for access.** International research shows a *distance decay effect*: people living closer to service facilities are more likely to use them. In a Swiss mental health system, travel time by public transport to the nearest facility negatively predicted the use of outpatient services for all mental disorders, while no such effect was found for inpatient wards, except for organic mental disorders [8]. The authors concluded that outpatient clinics should be located decentralised and in the largest communities, as close as possible to where people live [8]. In other words, *where* outpatient care is located directly affects *who* receives it.

## 2. Research Question

> **Which neighbourhoods in Almere combine high mental health care needs with poor access to existing outpatient mental health facilities, and where would a new facility add the most value?**

**Sub-questions:**

1. How are existing mental health care facilities distributed across Almere?
2. Which neighbourhoods show the highest indicators of mental health need (e.g. loneliness, psychological complaints, financial stress, demographic risk profile)?
3. How accessible are existing facilities from each neighbourhood (distance and public transport)?
4. Which neighbourhoods score highest on a combined need-versus-access index, and how robust is this ranking to different weighting choices?

## 3. Scope

- **Area:** municipality of Almere, at district (*wijk*) and neighbourhood (*buurt*) level
- **Focus:** outpatient mental health care, where distance decay effects are strongest
- **Data:** public sources only (CBS, RIVM, GGD Flevoland, PDOK, OpenStreetMap)
- **Purpose:** an exploratory, decision-support analysis. This is not a causal study and not a substitute for the local expertise of care providers and the municipality

## References

1. RIVM – Mentale Gezondheid Monitor, *Enige psychische aandoening* (NEMESIS-3). [Link](https://www.rivm.nl/mentale-gezondheid/monitor/algemene-bevolking/enige-psychische-aandoening)
2. *Anxiety and mood disorders on the rise: exploring clinical profiles and risk factors in the Netherlands*, European Psychiatry. [Link](https://resolve.cambridge.org/core/journals/european-psychiatry/article/anxiety-and-mood-disorders-on-the-rise-exploring-clinical-profiles-and-risk-factors-in-the-netherlands/E7C962D5DA4E8CDE5F2AE047C5A8B051)
3. *Prevalence and trends of common mental disorders from 2007–2009 to 2019–2022: results from NEMESIS*. [Link](https://pure.eur.nl/en/publications/prevalence-and-trends-of-common-mental-disorders-from-2007-2009-t/)
4. Radar (AVROTROS) & MIND – *Wachttijden bij de ggz lopen op tot twee jaar*. [Link](https://radar.avrotros.nl/artikel/wachttijden-ggz-lopen-op-tot-twee-jaar-62372)
5. GGD Flevoland – *Gezondheidsmonitor Volwassenen en Ouderen 2022, alle gemeenten (volwassenen)*. [Link](https://ggdflevoland.nl/app/uploads/sites/6/2024/12/Gezondheidsmonitor-Volwassenen-en-Ouderen-2022-Gemeenten-Volwassenen-versie1.pdf)
6. Gemeente Almere – *Ontwerp Omgevingsvisie Almere 2050* (2025). [Link](https://pas.commissiemer.nl/files/nl/3768/08c-bl-ontwerp-omgevingsvisie-almere-2050-ontwerp-omgevingsvisie.pdf)
7. Binnenlands Bestuur – *Coalitiepartijen: 25.000 woningen in Almere*. [Link](https://www.binnenlandsbestuur.nl/ruimte-en-milieu/privacy/coalitiepartijen-25-000-woningen-in-almere)
8. Stulz, N., Pichler, E.-M., Kawohl, W., & Hepp, U. (2018). *The gravitational force of mental health services: distance decay effects in a rural Swiss service area*. BMC Health Services Research. [Link](https://pmc.ncbi.nlm.nih.gov/articles/PMC5800291)

In [1]:
#first we import the required libraries

# Data handling
import pandas as pd
import numpy as np

# Data sources
import cbsodata
import requests

# Geospatial
import geopandas as gpd

# Visualisation
import matplotlib.pyplot as plt
import folium

# Scoring
from sklearn.preprocessing import MinMaxScaler

# Settings
pd.set_option("display.max_columns", None)  # show all columns when inspecting data

# Constants
GM_ALMERE = "GM0034"

Next, import databases we will use

In [7]:
# Load RIVM data from CBS API

rivm_raw = pd.DataFrame(
    cbsodata.get_data("50150NED", catalog_url="dataderden.cbs.nl")
)
print(rivm_raw.shape)
rivm_raw.head()

rivm_raw.to_parquet("data/rivm_raw.parquet")

(823950, 65)


In [8]:
# now we load CBS-kerncijfers data from CBS API

kern_raw = pd.DataFrame(
    cbsodata.get_data("85984NED"))
print(kern_raw.shape)
kern_raw.head()

kern_raw.to_parquet("data/kern_raw.parquet")


(18310, 123)


In [4]:
# now we will load the neighborhood boundaries from PDOK 

buurten_raw = gpd.read_file("data/wijkenbuurten_2024.gpkg", layer="buurten")
print(buurten_raw.shape)
buurten_raw.head()

(14668, 245)


,buurtcode,buurtnaam,wijkcode,gemeentecode,gemeentenaam,indelingswijziging_wijken_en_buurten,water,meest_voorkomende_postcode,dekkingspercentage,omgevingsadressendichtheid,stedelijkheid_adressen_per_km2,bevolkingsdichtheid_inwoners_per_km2,aantal_inwoners,mannen,vrouwen,percentage_personen_0_tot_15_jaar,percentage_personen_15_tot_25_jaar,percentage_personen_25_tot_45_jaar,percentage_personen_45_tot_65_jaar,percentage_personen_65_jaar_en_ouder,percentage_ongehuwd,percentage_gehuwd,percentage_gescheid,percentage_verweduwd,geboorte_totaal,geboortes_per_1000_inwoners,sterfte_totaal,sterfte_relatief,aantal_huishoudens,percentage_eenpersoonshuishoudens,percentage_huishoudens_zonder_kinderen,percentage_huishoudens_met_kinderen,gemiddelde_huishoudsgrootte,percentage_met_herkomstland_nederland,percentage_met_herkomstland_uit_europa_excl_nl,percentage_met_herkomstland_buiten_europa,percentage_geb_in_nl_met_herkomstland_nederland,perc_geb_in_nl_met_herkomstland_in_europa_ex_nl,perc_geb_in_nl_met_herkomstland_buiten_europa,perc_geb_buiten_nl_met_herkomstlnd_in_europa_ex_nl,perc_geb_buiten_nl_met_herkomstlnd_buiten_europa,aantal_bedrijven_landbouw_bosbouw_visserij,aantal_bedrijven_nijverheid_energie,aantal_bedrijven_handel_en_horeca,aantal_bedrijven_vervoer_informatie_communicatie,aantal_bedrijven_financieel_onroerend_goed,aantal_bedrijven_zakelijke_dienstverlening,aantal_bedrijven_overheid_onderwijs_en_zorg,aantal_bedrijven_cultuur_recreatie_overige,aantal_bedrijfsvestigingen,woningvoorraad,gemiddelde_woningwaarde,percentage_eengezinswoning,percentage_meergezinswoning,percentage_bewoond,percentage_koopwoningen,percentage_huurwoningen,perc_huurwoningen_in_bezit_woningcorporaties,perc_huurwoningen_in_bezit_overige_verhuurders,percentage_woningen_met_eigendom_onbekend,percentage_bouwjaarklasse_tot_2000,percentage_bouwjaarklasse_vanaf_2000,percentage_leegstand_woningen,gemiddeld_gasverbruik_totaal,gemiddeld_gasverbruik_appartement,gemiddeld_gasverbruik_tussenwoning,gemiddeld_gasverbruik_hoekwoning,gemiddeld_gasverbruik_2_onder_1_kap_woning,gemiddeld_gasverbruik_vrijstaande_woning,gemiddeld_gasverbruik_huurwoning,gemiddeld_gasverbruikkoopwoning,gemiddeld_elektriciteitsverbruik_totaal,gemiddeld_elektriciteitsverbruik_appartement,gemiddeld_elektriciteitsverbruik_tussenwoning,gemiddeld_elektriciteitsverbruik_hoekwoning,gem_elektriciteitsverbruik_2_onder_1_kap_woning,gem_elektriciteitsverbruik_vrijstaande_woning,gemiddeld_elektriciteitsverbruik_huurwoning,gemiddeld_elektriciteitsverbruikkoopwoning,percentage_woningen_met_stadsverwarming,opleidingsniveau_laag,opleidingsniveau_middelbaar,opleidingsniveau_hoog,netto_arbeidsparticipatie,percentage_werknemers,percentage_zelfstandigen,aantal_inkomensontvangers,gemiddeld_inkomen_per_inkomensontvanger,gemiddeld_inkomen_per_inwoner,percentage_personen_met_hoog_inkomen,percentage_personen_met_laag_inkomen,percentage_huishoudens_met_hoog_inkomen,percentage_huishoudens_met_laag_inkomen,percentage_huishoudens_met_laagste_inkomen,percentage_huishoudens_onder_of_rond_sociaal_minimum,aantal_personen_met_een_ao_uitkering_totaal,aantal_personen_met_een_ww_uitkering_totaal,aantal_personen_met_een_alg_bijstandsuitkering_tot,aantal_personen_met_een_aow_uitkering_totaal,gemiddeld_gestandaardiseerd_inkomen_van_huishoudens,huishoudens_tot_110_percent_van_sociaal_minimum,huishoudens_tot_120_percent_van_sociaal_minimum,mediaan_vermogen_van_particuliere_huish,aantal_jongeren_met_jeugdzorg_in_natura,percentage_jongeren_met_jeugdzorg_in_natura,aantal_wmo_clienten,aantal_wmo_clienten_per_1000_inwoners,personenautos_totaal,personenautos_per_huishouden,personenautos_per_km2,motortweewielers_totaal,aantal_personenautos_met_brandstof_benzine,aantal_personenautos_met_overige_brandstof,oppervlakte_totaal_in_ha,oppervlakte_land_in_ha,oppervlakte_water_in_ha,huisartsenpraktijk_gemiddelde_afstand_in_km,huisartsenpraktijk_gemiddeld_aantal_binnen_1_km,huisartsenpraktijk_gemiddeld_aantal_binnen_3_km,huisartsenpraktijk_gemiddeld_aantal_bin

In [9]:
# --- OpenStreetMap: mental health care locations & public transport in Almere ---

# Two public Overpass servers: if the first fails, try the second
OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]

# Identify ourselves to the server (prevents the 406 error)
HEADERS = {
    "User-Agent": "almere-mental-health-access/1.0 (portfolio project; contact@optivueconsultancy.nl)",
    "Accept": "application/json",
}


def run_overpass(query: str) -> list:
    """Send a query to Overpass and return the found map elements."""
    for url in OVERPASS_URLS:
        try:
            resp = requests.post(url, data={"data": query}, headers=HEADERS, timeout=90)
            resp.raise_for_status()  # stop here if the server returns an error
            return resp.json()["elements"]  # results are stored under "elements"
        except requests.RequestException as e:
            print(f"Failed on {url}: {e}")  # try the next server
    raise RuntimeError("All Overpass servers failed.")


def elements_to_df(elements: list) -> pd.DataFrame:
    """Turn Overpass results into a DataFrame with name, type and coordinates."""
    return pd.DataFrame([{
        "osm_id": e["id"],
        "osm_type": e["type"],  # node (point), way (shape) or relation (group)
        "name": e.get("tags", {}).get("name"),
        "healthcare": e.get("tags", {}).get("healthcare"),
        "speciality": e.get("tags", {}).get("healthcare:speciality"),
        # Points have lat/lon, shapes have a "center" -> take whichever exists
        "lat": e.get("lat", e.get("center", {}).get("lat")),
        "lon": e.get("lon", e.get("center", {}).get("lon")),
    } for e in elements])


# Query 1: mental health care locations within Almere (admin_level 8 = municipality)
ggz_query = """
[out:json][timeout:60];
area["name"="Almere"]["admin_level"="8"]->.a;
(
  nwr["healthcare"="psychotherapist"](area.a);
  nwr["healthcare"="counselling"](area.a);
  nwr["healthcare:speciality"~"psychiatry"](area.a);
);
out center tags;
"""

ggz_raw = elements_to_df(run_overpass(ggz_query))
print(f"Mental health care locations found: {len(ggz_raw)}")


# Query 2: bus stops and train stations (for accessibility later)
ov_query = """
[out:json][timeout:60];
area["name"="Almere"]["admin_level"="8"]->.a;
(
  nwr["highway"="bus_stop"](area.a);
  nwr["railway"="station"](area.a);
);
out center tags;
"""

ov_raw = elements_to_df(run_overpass(ov_query))
print(f"Public transport stops found: {len(ov_raw)}")


# Save raw data locally so we don't need to query the API again
ggz_raw.to_parquet("data/ggz_raw.parquet")
ov_raw.to_parquet("data/ov_raw.parquet")

ggz_raw.head()

Mental health care locations found: 3
Public transport stops found: 274


,osm_id,osm_type,name,healthcare,speciality,lat,lon
0,2839701318,node,PsyQ Almere,psychotherapist,physiotherapy;psychotherapist,52.374577,5.221066
1,2840408473,node,R. Bouwman Psychiatrie Almere,psychotherapist,NaN,52.379344,5.229228
2,14098730703,node,Feel Good Point,counselling,NaN,52.372704,5.228493


In [10]:
# Broader query: also search by tag variants and by name
ggz_query_broad = """
[out:json][timeout:60];
area["name"="Almere"]["admin_level"="8"]->.a;
(
  nwr["healthcare"~"psychotherapist|counselling|psychologist"](area.a);
  nwr["healthcare:speciality"~"psychiatry|psychology|psychotherapy"](area.a);
  nwr["name"~"ggz|psycholo|psychiat|psychotherap",i](area.a);  // i = case-insensitive
);
out center tags;
"""

ggz_osm = elements_to_df(run_overpass(ggz_query_broad))
print(f"Mental health care locations found (broad query): {len(ggz_osm)}")
ggz_osm[["name", "healthcare", "speciality"]]

Mental health care locations found (broad query): 5


,name,healthcare,speciality
0,PsyQ Almere,psychotherapist,physiotherapy;psychotherapist
1,Pedagogisch en psychologisch centrum Antwoord ...,NaN,NaN
2,R. Bouwman Psychiatrie Almere,psychotherapist,NaN
3,Feel Good Point,counselling,NaN
4,Forensisch Psychiatrisch Centrum De Oostvaarde...,NaN,NaN


In [11]:
# Classify each OSM location: include in analysis or not, and why
ggz_osm["include"] = [True, None, True, None, False]  # None = still to verify
ggz_osm["reason"] = [
    "Outpatient mental health care",
    "To verify: likely youth/family care",
    "Psychiatry practice",
    "To verify: counselling may be coaching, not mental health care",
    "Forensic psychiatric centre: not publicly accessible outpatient care",
]

ggz_osm[["name", "include", "reason"]]

,name,include,reason
0,PsyQ Almere,True,Outpatient mental health care
1,Pedagogisch en psychologisch centrum Antwoord ...,None,To verify: likely youth/family care
2,R. Bouwman Psychiatrie Almere,True,Psychiatry practice
3,Feel Good Point,None,"To verify: counselling may be coaching, not me..."
4,Forensisch Psychiatrisch Centrum De Oostvaarde...,False,Forensic psychiatric centre: not publicly acce...


Because we did not find all of the different healthcare providers using OSM, i have manually created a CSV doing research online with the full list i could find in Almere

### Mental health care locations: geocoding

The verified list of mental health care locations (`ggz_manual.csv`) contains addresses only. These are converted to coordinates with the PDOK Locatieserver, the official Dutch address service based on the national address register (BAG).

In [12]:
import time

# Load the verified list of mental health care locations
ggz = pd.read_csv("data/ggz_manual.csv")

PDOK_URL = "https://api.pdok.nl/bzk/locatieserver/search/v3_1/free"


def geocode_pdok(address: str):
    """Look up an address in PDOK and return (lat, lon), or (None, None) if not found."""
    params = {"q": address, "fq": "type:adres", "rows": 1}  # only match real addresses
    resp = requests.get(PDOK_URL, params=params, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    docs = resp.json()["response"]["docs"]
    if not docs:
        return None, None
    # 'centroide_ll' looks like 'POINT(5.21 52.37)' -> lon first, then lat
    lon, lat = docs[0]["centroide_ll"].replace("POINT(", "").replace(")", "").split()
    return float(lat), float(lon)


# Geocode every address (short pause to be polite to the free API)
coords = []
for address in ggz["address"]:
    coords.append(geocode_pdok(address))
    time.sleep(0.2)

ggz[["lat", "lon"]] = pd.DataFrame(coords, index=ggz.index)

# Check: did every address get coordinates?
missing = ggz[ggz["lat"].isna()]
print(f"Geocoded: {ggz['lat'].notna().sum()} / {len(ggz)}")
if not missing.empty:
    print("Not found:")
    print(missing[["name", "address"]])

# Convert to a GeoDataFrame so we can map it and measure distances later
ggz_gdf = gpd.GeoDataFrame(
    ggz,
    geometry=gpd.points_from_xy(ggz["lon"], ggz["lat"]),
    crs="EPSG:4326",  # standard GPS coordinates
)

# Save for later steps
ggz_gdf.to_file("data/ggz_locations.gpkg", driver="GPKG")

ggz_gdf[["name", "include", "lat", "lon"]]

Geocoded: 19 / 19


,name,include,lat,lon
0,GGz Centraal - De Meregaard,TRUE,52.372133,5.240348
1,GGz Centraal - Gebiedsteam Almere West,TRUE,52.366549,5.187347
2,PsyQ Almere,TRUE,52.374289,5.219186
3,Max Ernst GGZ Almere (Volwassenen),TRUE,52.377311,5.220820
4,Mentaal Beter Almere - Middachtenlaan,TRUE,52.388312,5.269743
5,MoleMann Mental Health Almere,TRUE,52.377100,5.211220
6,Psychiatrisch Expertise Centrum,TRUE,52.396349,5.303665
7,Indigo Almere - Bartok kliniek,TRUE,52.367558,5.188446
8,HSK Almere,TRUE,52.375973,5.215554
9,Hoofdpersoon Psychologie Almere,TRUE,52.354282,5.149379
